# Exploratory Analysis: Plasma Arc Sensor Data

This notebook provides an initial exploration of multi-channel sensor recordings
used for plasma arc detection. The data consists of time-series measurements from
industrial plasma systems, collected during research at the IISc CST Department.

## Channels

Each recording contains 6 sensor channels:

| Channel | Description |
|---------|-------------|
| 0 | Voltage (V) |
| 1 | Current (A) |
| 2 | Spectral Band 1 - UV |
| 3 | Spectral Band 2 - Visible |
| 4 | Spectral Band 3 - Near-IR |
| 5 | Spectral Band 4 - Thermal IR |

Sampling rate: 10 kHz

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.data.preprocessing import SignalProcessor

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 4)

## 1. Load and Inspect Raw Data

Each `.npy` file contains a dictionary with keys:
- `signals`: shape `(num_timesteps, 6)`
- `labels`: shape `(num_timesteps,)` with 0 = normal, 1 = arc
- `severity`: shape `(num_timesteps,)` with float in [0, 1]

In [ ]:
# Generate synthetic data for demonstration
rng = np.random.default_rng(42)
sample_rate = 10_000
duration_sec = 2.0
num_samples = int(sample_rate * duration_sec)
num_channels = 6

t = np.linspace(0, duration_sec, num_samples)

# Baseline signals: low-frequency oscillations + noise
signals = np.column_stack([
    50 * np.sin(2 * np.pi * 50 * t) + rng.normal(0, 2, num_samples),   # Voltage
    10 * np.sin(2 * np.pi * 50 * t + 0.3) + rng.normal(0, 0.5, num_samples),  # Current
    rng.normal(0.5, 0.1, num_samples),  # UV
    rng.normal(0.3, 0.08, num_samples), # Visible
    rng.normal(0.2, 0.05, num_samples), # Near-IR
    rng.normal(0.1, 0.03, num_samples), # Thermal IR
]).astype(np.float32)

# Inject an arc event from t=0.8s to t=1.0s
arc_start = int(0.8 * sample_rate)
arc_end = int(1.0 * sample_rate)
signals[arc_start:arc_end, 0] += rng.normal(80, 20, arc_end - arc_start)   # Voltage spike
signals[arc_start:arc_end, 1] += rng.normal(30, 5, arc_end - arc_start)    # Current surge
signals[arc_start:arc_end, 2:] += rng.exponential(2, (arc_end - arc_start, 4))  # Spectral flash

labels = np.zeros(num_samples, dtype=np.int64)
labels[arc_start:arc_end] = 1

print(f'Signal shape: {signals.shape}')
print(f'Arc samples: {labels.sum()} / {len(labels)} ({100 * labels.mean():.1f}%)')

## 2. Visualize Raw Signals

Plot all 6 channels with the arc region highlighted.

In [ ]:
channel_names = ['Voltage', 'Current', 'UV', 'Visible', 'Near-IR', 'Thermal IR']

fig, axes = plt.subplots(num_channels, 1, figsize=(14, 12), sharex=True)
for ch, (ax, name) in enumerate(zip(axes, channel_names)):
    ax.plot(t, signals[:, ch], linewidth=0.3, alpha=0.8)
    ax.axvspan(t[arc_start], t[arc_end], alpha=0.2, color='red', label='Arc event')
    ax.set_ylabel(name, fontsize=10)
    if ch == 0:
        ax.legend(loc='upper right')

axes[-1].set_xlabel('Time (s)')
fig.suptitle('Multi-channel Sensor Data with Arc Event', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Signal Processing

Apply bandpass filtering and compute the RMS envelope to see
how preprocessing enhances arc visibility.

In [ ]:
proc = SignalProcessor(num_channels=6, sample_rate=sample_rate)

filtered = proc.bandpass_filter(signals)
envelope = proc.compute_rms_envelope(filtered, window_size=128)

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(t, filtered[:, 0], linewidth=0.3)
axes[0].axvspan(t[arc_start], t[arc_end], alpha=0.2, color='red')
axes[0].set_ylabel('Filtered Voltage')
axes[0].set_title('Bandpass Filtered Signal')

axes[1].plot(t, envelope[:, 0], linewidth=0.8, color='darkorange')
axes[1].axvspan(t[arc_start], t[arc_end], alpha=0.2, color='red')
axes[1].set_ylabel('RMS Envelope')
axes[1].set_xlabel('Time (s)')
axes[1].set_title('RMS Envelope (window=128 samples)')

plt.tight_layout()
plt.show()

## 4. Spectral Feature Extraction

Extract frequency-domain features per window: dominant frequency,
spectral centroid, bandwidth, and energy.

In [ ]:
spectral = proc.extract_spectral_features(signals, window_size=256)
print(f'Spectral features shape: {spectral.shape}')
print(f'Features per window: 4 per channel x 6 channels = {spectral.shape[1]}')

# Plot spectral energy for voltage channel across windows
voltage_energy = spectral[:, 3]  # 4th feature (energy) of channel 0
window_times = np.arange(len(voltage_energy)) * 256 / sample_rate

plt.figure(figsize=(14, 3))
plt.bar(window_times, voltage_energy, width=256/sample_rate * 0.9, alpha=0.7)
plt.axvspan(0.8, 1.0, alpha=0.2, color='red', label='Arc region')
plt.xlabel('Time (s)')
plt.ylabel('Spectral Energy')
plt.title('Voltage Channel Spectral Energy per Window')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Transient Detection

Simple derivative-based transient detection to identify potential arc initiation points.

In [ ]:
transients = proc.detect_transients(signals, threshold_sigma=3.0)

# Show transient detections on voltage channel
plt.figure(figsize=(14, 3))
plt.plot(t, signals[:, 0], linewidth=0.3, alpha=0.6, label='Voltage')
transient_times = t[transients[:, 0]]
transient_vals = signals[transients[:, 0], 0]
plt.scatter(transient_times, transient_vals, c='red', s=2, zorder=5, label='Transients')
plt.axvspan(0.8, 1.0, alpha=0.15, color='red')
plt.xlabel('Time (s)')
plt.ylabel('Voltage')
plt.title('Transient Detection on Voltage Channel')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Transients detected per channel: {transients.sum(axis=0)}')

## Summary

Key observations from the exploratory analysis:

1. **Arc events** produce clear spikes across voltage, current, and all spectral channels
2. **Bandpass filtering** removes low-frequency drift while preserving arc transients
3. **RMS envelope** provides a smooth indicator of arc intensity over time
4. **Spectral energy** in the voltage channel shows a distinct peak during arc events
5. **Transient detection** successfully identifies the onset and offset of arc events

These preprocessing steps feed directly into the CNN-LSTM model pipeline,
where the CNN extracts local patterns and the LSTM captures temporal dependencies
across the arc event lifecycle.